# My Game: Minotaur Dungeon

A compact dungeon adventure with four rooms, four weapon choices, a Minotaur fight, a restaurant, a treasure-room choice, one win state, and multiple lose states.


In [ ]:
from collections import defaultdict

In [24]:
class Game:
    def __init__(self, start_at):
        self.curr_location = start_at
        self.curr_location.has_been_visited = True
        self.inventory = {}
        self.print_commands = True

        self.hp = 2
        self.coins = 2
        self.weapon = None
        self.ally_weapon = None
        self.ally_hp = 1

        self.gate_open = False
        self.treasure_door_open = False
        self.minotaur_hp = 5
        self.minotaur_max_hp = 5
        self.minotaur_defeated = False
        self.monster_actions = 0
        self.next_target = "player"
        self.player_dodged = False
        self.ally_dodged = False
        self.dodged_minotaur = False

        self.princess_saved = False
        self.game_over = False
        self.previous_location = None

    def describe(self):
        self.describe_current_location()
        self.describe_status()
        self.describe_exits()
        self.describe_items()

    def describe_current_location(self):
        print(self.curr_location.description)

    def describe_status(self):
        weapon = self.weapon if self.weapon else "none"
        ally = self.ally_weapon if self.ally_weapon else "none"
        print(f"HP: {self.hp} | Coins: {self.coins} | Weapon: {weapon} | Ally: {ally}")
        if self.curr_location.name == "Monster Room" and not self.minotaur_defeated:
            print(f"Minotaur HP: {self.minotaur_hp}/{self.minotaur_max_hp}")

    def describe_exits(self):
        exits = [exit.capitalize() for exit in self.curr_location.connections.keys()]
        if exits:
            print("Exits: ", end="")
            print(*exits, sep=", ")

    def describe_items(self):
        if len(self.curr_location.items) > 0:
            print("You see:")
            for item_name in self.curr_location.items:
                item = self.curr_location.items[item_name]
                print(" * " + item.description)
                if self.print_commands:
                    for cmd in item.get_commands():
                        print("	" + cmd)

    def add_to_inventory(self, item):
        self.inventory[item.name] = item

    def is_in_inventory(self, item):
        return item.name in self.inventory

    def get_items_in_scope(self):
        items_in_scope = []
        for item_name in self.curr_location.items:
            items_in_scope.append(self.curr_location.items[item_name])
        for item_name in self.inventory:
            items_in_scope.append(self.inventory[item_name])
        return items_in_scope

    def lose(self, message):
        print(message)
        self.game_over = True
        return True

    def win(self, message):
        print(message)
        self.game_over = True
        return True


In [25]:
class Location:
    def __init__(self, name, description):
        self.name = name
        self.description = description
        self.properties = defaultdict(bool)
        self.connections = {}
        self.travel_descriptions = {}
        self.items = {}
        self.blocks = {}
        self.has_been_visited = False

    def set_property(self, property_name, property_bool=True):
        self.properties[property_name] = property_bool

    def get_property(self, property_name):
        return self.properties[property_name]

    def add_connection(self, direction, connected_location, travel_description=""):
        direction = direction.lower()
        self.connections[direction] = connected_location
        self.travel_descriptions[direction] = travel_description
        if direction == "north":
            connected_location.connections["south"] = self
        if direction == "south":
            connected_location.connections["north"] = self
        if direction == "east":
            connected_location.connections["west"] = self
        if direction == "west":
            connected_location.connections["east"] = self
        if direction == "up":
            connected_location.connections["down"] = self
        if direction == "down":
            connected_location.connections["up"] = self
        if direction == "in":
            connected_location.connections["out"] = self
        if direction == "out":
            connected_location.connections["in"] = self

    def add_item(self, name, item):
        self.items[name] = item

    def remove_item(self, item):
        self.items.pop(item.name)

    def is_blocked(self, direction, game):
        if direction not in self.blocks:
            return False
        block_description, preconditions = self.blocks[direction]
        return not check_preconditions(preconditions, game)

    def get_block_description(self, direction):
        if direction not in self.blocks:
            return ""
        block_description, _ = self.blocks[direction]
        return block_description

    def add_block(self, blocked_direction, block_description, preconditions):
        self.blocks[blocked_direction] = (block_description, preconditions)


In [26]:
def check_preconditions(preconditions, game, print_failure_reasons=True):
    all_conditions_met = True

    for check in preconditions:
        if check == "inventory_contains":
            item = preconditions[check]
            if not game.is_in_inventory(item):
                all_conditions_met = False
                if print_failure_reasons:
                    print("You don't have the %s." % item.name)

        if check == "gate_open":
            if not game.gate_open:
                all_conditions_met = False
                if print_failure_reasons:
                    print("The iron gate is closed. Try `open gate` first.")

        if check == "minotaur_defeated":
            if not game.minotaur_defeated:
                all_conditions_met = False
                if print_failure_reasons:
                    print("The Minotaur still blocks the treasure door.")

        if check == "dodged_minotaur":
            if not game.dodged_minotaur:
                all_conditions_met = False
                if print_failure_reasons:
                    print("You need to dodge at least one Minotaur attack before entering the Treasure Room.")

        if check == "treasure_door_open":
            if not game.treasure_door_open:
                all_conditions_met = False
                if print_failure_reasons:
                    print("The treasure door is shut. Try `open door`.")

    return all_conditions_met


In [27]:
class Item:
    def __init__(self, name, description, examine_text="", take_text="", start_at=None):
        self.name = name
        self.description = description
        self.examine_text = examine_text
        self.take_text = take_text if take_text else ("You take the %s." % self.name)
        self.properties = defaultdict(bool)
        self.properties["gettable"] = True
        self.commands = {}
        if start_at:
            start_at.add_item(name, self)

    def get_commands(self):
        return self.commands.keys()

    def set_property(self, property_name, property_bool=True):
        self.properties[property_name] = property_bool

    def get_property(self, property_name):
        return self.properties[property_name]

    def add_action(self, command_text, function, arguments=(), preconditions={}):
        self.commands[command_text] = (function, arguments, preconditions)

    def do_action(self, command_text, game):
        if command_text in self.commands:
            function, arguments, preconditions = self.commands[command_text]
            if check_preconditions(preconditions, game):
                return function(game, arguments)
            return False
        print("Cannot perform the action %s." % command_text)
        return False


In [28]:
class Parser:
    def __init__(self, game):
        self.command_history = []
        self.game = game

    def get_player_intent(self, command):
        command = command.lower().strip()
        if "," in command:
            return "sequence"
        if self.get_direction(command):
            return "direction"
        if command == "look" or command == "l":
            return "redescribe"
        if "examine " in command or command.startswith("x "):
            return "examine"
        if "take " in command or "get " in command:
            return "take"
        if "drop " in command:
            return "drop"
        if "inventory" in command or command == "i":
            return "inventory"
        for item in self.game.get_items_in_scope():
            for special_command in item.get_commands():
                if command == special_command.lower():
                    return "special"
        return None

    def parse_command(self, command):
        self.command_history.append(command)
        command = command.lower().strip()
        end_game = False
        intent = self.get_player_intent(command)

        if intent == "direction":
            end_game = self.go_in_direction(command)
        elif intent == "redescribe":
            self.game.describe()
        elif intent == "examine":
            self.examine(command)
        elif intent == "take":
            end_game = self.take(command)
        elif intent == "drop":
            self.drop(command)
        elif intent == "inventory":
            self.check_inventory(command)
        elif intent == "special":
            end_game = self.run_special_command(command)
        elif intent == "sequence":
            end_game = self.execute_sequence(command)
        else:
            print("I'm not sure what you want to do.")

        return end_game or self.game.game_over

    def go_in_direction(self, command):
        direction = self.get_direction(command)
        if direction and direction in self.game.curr_location.connections:
            if self.game.curr_location.is_blocked(direction, self.game):
                print(self.game.curr_location.get_block_description(direction))
                return False

            old_location = self.game.curr_location
            self.game.previous_location = old_location
            self.game.curr_location = self.game.curr_location.connections[direction]

            if old_location.name == "Monster Room" and self.game.curr_location.name != "Monster Room" and not self.game.minotaur_defeated:
                self.game.minotaur_max_hp += 1
                self.game.minotaur_hp = self.game.minotaur_max_hp
                print("The Minotaur roars and grows stronger while you retreat.")

            self.game.describe()
        else:
            print("You can't go %s from here." % direction.capitalize())
        return False

    def check_inventory(self, command):
        if len(self.game.inventory) == 0:
            print("You don't have anything.")
        else:
            descriptions = [item.description for item in self.game.inventory.values()]
            print("You have: " + ", ".join(descriptions))
        print(f"HP: {self.game.hp} | Coins: {self.game.coins}")

    def examine(self, command):
        matched_item = False
        for item_name in self.game.curr_location.items:
            if item_name in command:
                item = self.game.curr_location.items[item_name]
                print(item.examine_text if item.examine_text else item.description)
                matched_item = True
                break
        for item_name in self.game.inventory:
            if item_name in command:
                item = self.game.inventory[item_name]
                print(item.examine_text if item.examine_text else item.description)
                matched_item = True
        if not matched_item:
            print("You don't see anything special.")

    def take(self, command):
        matched_item = False
        for item_name in list(self.game.curr_location.items.keys()):
            if item_name in command:
                item = self.game.curr_location.items[item_name]
                if item.get_property("gettable"):
                    self.game.add_to_inventory(item)
                    self.game.curr_location.remove_item(item)
                    print(item.take_text)
                    return item.get_property("end_game")
                print("You cannot take the %s." % item_name)
                matched_item = True
                break
        if not matched_item:
            for item_name in self.game.inventory:
                if item_name in command:
                    print("You already have the %s." % item_name)
                    matched_item = True
        if not matched_item:
            print("You can't find it.")
        return False

    def drop(self, command):
        for item_name in list(self.game.inventory.keys()):
            if item_name in command:
                item = self.game.inventory[item_name]
                self.game.curr_location.add_item(item_name, item)
                self.game.inventory.pop(item_name)
                print("You drop the %s." % item_name)
                return
        print("You don't have that.")

    def run_special_command(self, command):
        for item in self.game.get_items_in_scope():
            for special_command in item.get_commands():
                if command == special_command.lower():
                    return item.do_action(special_command, self.game)
        return False

    def execute_sequence(self, command):
        for cmd in command.split(","):
            if self.game.game_over:
                return True
            end_game = self.parse_command(cmd.strip())
            if end_game:
                return True
        return False

    def get_direction(self, command):
        command = command.lower().strip()
        if command == "n" or "north" in command:
            return "north"
        if command == "s" or "south" in command:
            return "south"
        if command == "e" or "east" in command:
            return "east"
        if command == "w" or "west" in command:
            return "west"
        if command == "up" or command == "go up":
            return "up"
        if command == "down" or command == "go down":
            return "down"
        if command.startswith("go out"):
            return "out"
        if command.startswith("go in"):
            return "in"
        for exit in self.game.curr_location.connections.keys():
            if command == exit.lower() or command == "go " + exit.lower():
                return exit
        return None


In [29]:
WEAPONS = ["sword", "spear", "magic", "arrow"]
DAMAGE_WEAPONS = ["sword", "spear"]


def choose_weapon(game, args):
    weapon = args[0]
    game.weapon = weapon
    print(f"You choose the {weapon}.")
    return False


def team_up(game, args):
    weapon = args[0]
    game.ally_weapon = weapon
    game.ally_hp = 1
    print(f"A teammate carrying a {weapon} joins you.")
    return False


def open_gate(game, args):
    game.gate_open = True
    print("You push the iron gate open. The Monster Room waits to the south.")
    return False


def check_hp(game, args):
    print(f"HP: {game.hp} | Coins: {game.coins} | Minotaur HP: {game.minotaur_hp}/{game.minotaur_max_hp}")
    return False


def monster_counterattack(game):
    if game.curr_location.name != "Monster Room" or game.minotaur_defeated:
        return False

    game.monster_actions += 1
    if game.monster_actions % 2 == 1:
        print(f"The Minotaur lowers its horns. It will attack the {game.next_target} next.")
        return False

    target = game.next_target
    if target == "player":
        if game.player_dodged:
            print("You dodge the Minotaur's charge.")
            game.dodged_minotaur = True
        else:
            game.hp -= 1
            print(f"The Minotaur hits you. HP is now {game.hp}.")
        game.player_dodged = False
        game.next_target = "ally"
    else:
        if not game.ally_weapon or game.ally_hp <= 0:
            print("The Minotaur finds no teammate to strike.")
        elif game.ally_dodged:
            print(f"Your {game.ally_weapon} teammate dodges the Minotaur.")
        else:
            game.ally_hp = 0
            print(f"The Minotaur knocks out your {game.ally_weapon} teammate.")
        game.ally_dodged = False
        game.next_target = "player"

    if game.ally_weapon in DAMAGE_WEAPONS and game.ally_hp > 0 and not game.minotaur_defeated:
        game.minotaur_hp -= 1
        print(f"Your {game.ally_weapon} teammate strikes back. Minotaur HP is now {max(game.minotaur_hp, 0)}.")

    if game.hp <= 0:
        return game.lose("You collapse in the Monster Room. GAME OVER.")

    if game.minotaur_hp <= 0:
        game.minotaur_defeated = True
        print("The Minotaur falls. The treasure door can now be opened.")

    return False


def attack_minotaur(game, args):
    if game.minotaur_defeated:
        print("The Minotaur has already been defeated.")
        return False
    if not game.weapon:
        print("You need to choose a weapon first.")
        return False
    if game.weapon in DAMAGE_WEAPONS:
        game.minotaur_hp -= 1
        print(f"You attack with the {game.weapon}. Minotaur HP is now {max(game.minotaur_hp, 0)}.")
    else:
        print(f"Your {game.weapon} cannot hurt the Minotaur.")
    if game.minotaur_hp <= 0:
        game.minotaur_defeated = True
        print("The Minotaur falls. The treasure door can now be opened.")
        return False
    return monster_counterattack(game)


def dodge(game, args):
    game.player_dodged = True
    print("You prepare to dodge the next attack aimed at you.")
    return monster_counterattack(game)


def ally_dodge(game, args):
    weapon = args[0]
    if game.ally_weapon != weapon or game.ally_hp <= 0:
        print(f"You do not have an active {weapon} teammate.")
        return False
    game.ally_dodged = True
    print(f"You shout for your {weapon} teammate to dodge.")
    return monster_counterattack(game)


def open_treasure_door(game, args):
    if not game.minotaur_defeated:
        print("The Minotaur is still guarding the door.")
    elif not game.dodged_minotaur:
        print("You need to dodge at least one Minotaur attack before opening the treasure path.")
    else:
        game.treasure_door_open = True
        print("You open the heavy door to the Treasure Room.")
    return False


def drink_beer(game, args):
    if game.coins < 1:
        print("You do not have enough coins.")
    else:
        game.coins -= 1
        game.hp += 1
        print(f"You drink a bitter dungeon beer. HP: {game.hp}. Coins: {game.coins}.")
    return False


def eat_meal(game, args):
    if game.coins < 2:
        print("You do not have enough coins.")
    else:
        game.coins -= 2
        game.hp += 2
        print(f"You eat a warm meal. HP: {game.hp}. Coins: {game.coins}.")
    return False


def go_back(game, args):
    if game.previous_location is None:
        print("You are not sure where to go back to.")
    else:
        current = game.curr_location
        game.curr_location = game.previous_location
        game.previous_location = current
        game.describe()
    return False


def choose_princess(game, args):
    game.princess_saved = True
    game.coins += 10
    print("You choose the princess. She thanks you, and the treasure room grants you 10 coins.")
    return False


def choose_treasure_box(game, args):
    return game.lose("The treasure box opens its jaws and eats you. GAME OVER.")


def eat_with_princess(game, args):
    if not game.princess_saved:
        print("The princess is not here yet.")
        return False
    return game.win("You share a victory meal with the princess. YOU WIN!")


In [30]:
def build_game():
    # Locations
    starting_room = Location("Starting Room", "You stand before a sealed dungeon gate. A weapon rack waits beside you.")
    monster_room = Location("Monster Room", "You enter a stone arena. The Minotaur paws the ground and snorts.")
    restaurant = Location("Restaurant", "A warm underground restaurant glows with lantern light.")
    treasure_room = Location("Treasure Room", "Gold shines around a princess and a suspicious treasure box.")

    # Connections
    starting_room.add_connection("south", monster_room)
    starting_room.add_connection("east", restaurant)
    restaurant.add_connection("south", monster_room)

    # One-way dungeon exits. This avoids automatic reverse exits that would
    # let the player bypass the treasure-door puzzle.
    monster_room.connections["south"] = treasure_room
    monster_room.travel_descriptions["south"] = ""
    treasure_room.connections["west"] = monster_room
    treasure_room.travel_descriptions["west"] = ""
    treasure_room.connections["east"] = restaurant
    treasure_room.travel_descriptions["east"] = ""

    # Blocks / puzzles
    starting_room.add_block("south", "The closed iron gate blocks the way south.", {"gate_open": True})
    monster_room.add_block("south", "The treasure door will not open until you beat the Minotaur, dodge an attack, and open the door.", {
        "minotaur_defeated": True,
        "dodged_minotaur": True,
        "treasure_door_open": True,
    })

    # Starting room tools
    weapon_rack = Item("weapon rack", "a rack holding a sword, spear, magic wand, and bow", "You can choose sword, spear, magic, or arrow.", start_at=starting_room)
    weapon_rack.set_property("gettable", False)
    gate = Item("gate", "a closed iron gate", "The gate leads south into the Monster Room.", start_at=starting_room)
    gate.set_property("gettable", False)

    for weapon in WEAPONS:
        weapon_rack.add_action(f"choose {weapon}", choose_weapon, (weapon,))
        weapon_rack.add_action(f"team up with {weapon}", team_up, (weapon,))
    gate.add_action("open gate", open_gate)

    # Inventory status item for global helper commands
    status = Item("status", "an adventurer status card", "It tracks your HP, coins, and fight progress.")
    status.add_action("check hp", check_hp)

    # Monster room
    minotaur = Item("minotaur", "a massive Minotaur with bronze horns", "Only sword and spear attacks can wound it.", start_at=monster_room)
    minotaur.set_property("gettable", False)
    minotaur.add_action("attack", attack_minotaur)
    minotaur.add_action("dodge", dodge)
    minotaur.add_action("open door", open_treasure_door)
    for weapon in WEAPONS:
        minotaur.add_action(f"{weapon} dodge", ally_dodge, (weapon,))

    # Restaurant
    menu = Item("menu", "a restaurant menu", "Beer costs 1 coin. A meal costs 2 coins.", start_at=restaurant)
    menu.set_property("gettable", False)
    menu.add_action("drink beer", drink_beer)
    menu.add_action("drink bear", drink_beer)
    menu.add_action("eat meal", eat_meal)
    menu.add_action("eat with princess", eat_with_princess)
    menu.add_action("go back", go_back)
    for weapon in WEAPONS:
        menu.add_action(f"change {weapon}", choose_weapon, (weapon,))

    # Treasure room
    princess = Item("princess", "a brave princess waiting beside the treasure", "She looks ready to leave this dungeon.", start_at=treasure_room)
    princess.set_property("gettable", False)
    princess.add_action("choose princess", choose_princess)
    treasure_box = Item("treasure box", "a glittering treasure box", "It seems to breathe.", start_at=treasure_room)
    treasure_box.set_property("gettable", False)
    treasure_box.add_action("choose treasure box", choose_treasure_box)

    game = Game(starting_room)
    game.add_to_inventory(status)
    return game


## Play the game

Run the next cell and try the playthrough below:

```text
choose sword
team up with spear
open gate
go south
attack
dodge
attack
spear dodge
attack
open door
go south
choose princess
go east
eat with princess
```


In [ ]:
def game_loop():
    game = build_game()
    parser = Parser(game)
    game.describe()

    command = ""
    while not (command.lower() == "exit" or command.lower() == "q"):
        command = input(">")
        end_game = parser.parse_command(command)
        if end_game:
            return

# game_loop()

## Quick automated test

This cell runs the winning playthrough without interactive input.


In [32]:
game = build_game()
parser = Parser(game)
game.describe()

commands = [
    "choose sword",
    "team up with spear",
    "open gate",
    "go south",
    "attack",
    "dodge",
    "attack",
    "spear dodge",
    "attack",
    "open door",
    "go south",
    "choose princess",
    "go east",
    "eat with princess",
]

for command in commands:
    print(f"\n> {command}")
    if parser.parse_command(command):
        break


You stand before a sealed dungeon gate. A weapon rack waits beside you.
HP: 2 | Coins: 2 | Weapon: none | Ally: none
Exits: South, East
You see:
 * a rack holding a sword, spear, magic wand, and bow
	choose sword
	team up with sword
	choose spear
	team up with spear
	choose magic
	team up with magic
	choose arrow
	team up with arrow
 * a closed iron gate
	open gate

> choose sword
You choose the sword.

> team up with spear
A teammate carrying a spear joins you.

> open gate
You push the iron gate open. The Monster Room waits to the south.

> go south
You enter a stone arena. The Minotaur paws the ground and snorts.
HP: 2 | Coins: 2 | Weapon: sword | Ally: spear
Minotaur HP: 5/5
Exits: North, South
You see:
 * a massive Minotaur with bronze horns
	attack
	dodge
	open door
	sword dodge
	spear dodge
	magic dodge
	arrow dodge

> attack
You attack with the sword. Minotaur HP is now 4.
The Minotaur lowers its horns. It will attack the player next.

> dodge
You prepare to dodge the next atta